# Chapter 2 — Coding Our First Neurons

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saanikakulkarni07/learn-neural-nets/blob/main/ch02-coding-our-first-neurons/exercises.ipynb)

This is the first **code** chapter. We'll build a single neuron in pure Python, scale it to a layer, then rewrite it all with NumPy — ending at the one line that powers every dense layer in the book:

```python
layer_outputs = np.dot(inputs, np.array(weights).T) + biases
```

**Goals**
1. Compute a single neuron, then a layer, in plain Python.
2. See that a neuron's pre-bias output *is* a dot product.
3. Vectorize a layer with NumPy.
4. Handle a **batch** of samples and get the **transpose** right (the #1 source of bugs).
5. Push a small astro 'catalog' through a layer.


## 0. Setup

In [ ]:
import numpy as np
print('numpy', np.__version__)

## 1. Concept check (no code)

1. A single neuron has 5 inputs. How many **weights** does it have? How many **biases**?
2. A dense layer has 4 inputs and 3 neurons. How many weights total? How many biases?
3. What's the difference between a **dot product** and a **matrix product** (what does each return)?
4. `inputs` is shape `(3, 4)` and `weights` is shape `(3, 4)`. Why can't we do `np.dot(inputs, weights)` directly, and what fixes it?
5. After `np.dot(inputs, weights.T)`, what does **row i** of the result represent?


_Your answers:_

1. 
2. 
3. 
4. 
5. 

<details><summary>Check yourself</summary>

1. 5 weights, 1 bias.
2. 4×3 = 12 weights, 3 biases (one per neuron).
3. Dot product → a scalar; matrix product → a matrix (of dot products of rows×columns).
4. Inner dimensions must match: `(3,4)·(3,4)` fails. Transpose the second → `(3,4)·(4,3)→(3,3)`.
5. The layer's outputs for **sample i** (rows = samples, columns = neurons).
</details>

## 2. A single neuron (pure Python)

**Exercise 2.** A neuron with 4 inputs. Compute its output as `sum(input*weight) + bias`, no NumPy. Should be `4.8`.

In [ ]:
inputs  = [1.0, 2.0, 3.0, 2.5]
weights = [0.2, 0.8, -0.5, 1.0]
bias    = 2.0

output = sum(i * w for i, w in zip(inputs, weights)) + bias
print(output)
assert abs(output - 4.8) < 1e-9
print('✓')

## 3. A layer of neurons (pure Python)

Three neurons, same inputs, each with its own weights + bias. Use nested loops with `zip`. Expected `[4.8, 1.21, 2.385]`.

In [ ]:
inputs  = [1, 2, 3, 2.5]
weights = [[0.2, 0.8, -0.5, 1.0],
           [0.5, -0.91, 0.26, -0.5],
           [-0.26, -0.27, 0.17, 0.87]]
biases  = [2.0, 3.0, 0.5]

layer_outputs = []
for neuron_weights, neuron_bias in zip(weights, biases):
    neuron_output = 0
    for n_input, weight in zip(inputs, neuron_weights):
        neuron_output += n_input * weight
    neuron_output += neuron_bias
    layer_outputs.append(neuron_output)

print(layer_outputs)
assert np.allclose(layer_outputs, [4.8, 1.21, 2.385])
print('✓')

## 4. The dot product

A neuron's pre-bias output is a dot product: $\vec a \cdot \vec b = \sum_i a_i b_i$.

**Exercise 4a.** Compute `[1,2,3]·[2,3,4]` by hand-loop *and* with `np.dot`; confirm both give `20`.

In [ ]:
a = [1, 2, 3]
b = [2, 3, 4]

by_hand = sum(ai * bi for ai, bi in zip(a, b))
by_numpy = np.dot(a, b)
print(by_hand, by_numpy)
assert by_hand == by_numpy == 20
print('✓ a neuron is just this + a bias')

## 5. Single neuron, then a layer, with NumPy

**Exercise 5.** Reproduce the single neuron (`4.8`) and the 3-neuron layer (`[4.8, 1.21, 2.385]`) using `np.dot`. For one input vector, weights can stay as the `(neurons, inputs)` matrix.

In [ ]:
inputs  = [1.0, 2.0, 3.0, 2.5]
weights = [[0.2, 0.8, -0.5, 1.0],
           [0.5, -0.91, 0.26, -0.5],
           [-0.26, -0.27, 0.17, 0.87]]
biases  = [2.0, 3.0, 0.5]

# single neuron = first weight vector + first bias
single = np.dot(weights[0], inputs) + biases[0]
print('single neuron:', single)
assert np.isclose(single, 4.8)

# whole layer: matrix (neurons x inputs) dot vector -> one output per neuron
layer = np.dot(weights, inputs) + biases
print('layer:', layer)
assert np.allclose(layer, [4.8, 1.21, 2.385])
print('✓')

## 6. ⭐ A batch of samples + the transpose

Now feed **3 samples** at once. `inputs` is `(3 samples, 4 features)`, `weights` is `(3 neurons, 4 features)`. Inner dims (4 and 3... wait — 4 vs 4 on the *wrong* axes) don't line up for the matrix product, so we transpose weights to `(4, 3)`.

**Exercise 6.** Fill the `.T`. Check the shape is `(3, 3)` and matches the book's output.

In [ ]:
inputs  = [[1.0, 2.0, 3.0, 2.5],
           [2.0, 5.0, -1.0, 2.0],
           [-1.5, 2.7, 3.3, -0.8]]      # (3 samples, 4 features)
weights = [[0.2, 0.8, -0.5, 1.0],
           [0.5, -0.91, 0.26, -0.5],
           [-0.26, -0.27, 0.17, 0.87]]  # (3 neurons, 4 features)
biases  = [2.0, 3.0, 0.5]

print('inputs shape :', np.array(inputs).shape)
print('weights shape:', np.array(weights).shape)
print('weights.T    :', np.array(weights).T.shape)

layer_outputs = np.dot(inputs, np.array(weights).T) + biases
print('output shape :', layer_outputs.shape)
print(layer_outputs)

expected = [[ 4.8,   1.21,  2.385],
            [ 8.9,  -1.81,  0.2  ],
            [ 1.41,  1.051, 0.026]]
assert layer_outputs.shape == (3, 3)
assert np.allclose(layer_outputs, expected)
print('✓ row i = layer outputs for sample i')

**Why `inputs` first?** Try swapping to `np.dot(np.array(weights), np.array(inputs).T)` and look at the shape — you'd get `(neurons, samples)`, i.e. transposed. We want rows = samples so the next layer can consume the batch directly. Run to see the difference:

In [ ]:
other = np.dot(np.array(weights), np.array(inputs).T) + np.array(biases).reshape(-1, 1)
print('neuron-major (samples are columns):')
print(other)
print('shape', other.shape, '-> this is the transpose of what we want')
assert np.allclose(other.T, layer_outputs)

## 7. Shapes drill

**Exercise 7.** Predict each shape *before* running. Fill `guess` for each; the asserts check you.

In [ ]:
lol   = [[1, 5, 6, 2], [3, 2, 1, 3]]
lolol = [[[1,5,6,2],[3,2,1,3]], [[5,2,1,2],[6,4,8,4]], [[2,8,5,3],[1,1,9,4]]]

guess_lol   = (2, 4)      # TODO predict
guess_lolol = (3, 2, 4)   # TODO predict

assert np.array(lol).shape   == guess_lol,   np.array(lol).shape
assert np.array(lolol).shape == guess_lolol, np.array(lolol).shape
print('✓ shapes:', np.array(lol).shape, np.array(lolol).shape)

## 8. 🔭 Astro bonus — a catalog through a layer

Treat a batch as a tiny **photometric catalog**: each row is a source, columns are magnitudes `[u, g, r, i]`. A dense layer of 2 neurons could be the first layer of a star/galaxy classifier. We'll hand-set weights so **neuron 0 computes the (u−g) color** and **neuron 1 computes (g−r)** — classic color features — to show a 'layer' is just learnable linear combinations of features.

**Exercise 8.** Set the two weight vectors so neuron 0 = `u - g` and neuron 1 = `g - r` (biases 0). Verify against direct column subtraction.

In [ ]:
# rows = sources, cols = [u, g, r, i] magnitudes
catalog = np.array([[18.1, 17.4, 16.9, 16.7],
                    [20.3, 19.9, 19.6, 19.5],
                    [15.2, 14.0, 13.1, 12.8]])   # (3 sources, 4 bands)

# neuron 0 -> u - g ;  neuron 1 -> g - r
weights = np.array([[ 1.0, -1.0, 0.0, 0.0],   # u - g
                    [ 0.0,  1.0, -1.0, 0.0]])  # g - r
biases = np.array([0.0, 0.0])

colors = np.dot(catalog, weights.T) + biases   # (3 sources, 2 colors)
print('colors [u-g, g-r] per source:')
print(colors)

u, g, r, i = catalog.T
assert np.allclose(colors[:, 0], u - g)
assert np.allclose(colors[:, 1], g - r)
print('✓ a layer = a batch of linear feature combinations; training just *learns* these weights')

### Stretch
- Add a 3rd neuron computing `r - i` and re-check the output shape `(3, 3)`.
- Time `np.dot` over a 100k-source catalog vs. a Python loop over rows — the vectorized speedup is the same reason we vectorize FITS-table operations.
- These hand-set 'color' weights are exactly what an optimizer would *discover* on its own in later chapters; here we just see that the machinery is plain linear algebra.